# 14장 LangSmith

이 장은 PDF 교재의 LangChain, RAG, DB 챗봇, LangGraph 실습을 바탕으로 LLM 애플리케이션을 관찰하고 디버깅하는 도구인 `LangSmith`를 정리합니다.

LLM 애플리케이션은 일반 프로그램보다 실행 흐름을 파악하기 어렵습니다. 프롬프트, 검색 결과, 모델 응답, 파서, 도구 호출, DB 조회가 여러 단계로 연결되기 때문입니다. LangSmith는 이런 실행 과정을 추적하고, 실패 원인을 찾고, 프롬프트와 체인을 개선하는 데 사용합니다.

## 이 장에서 다루는 내용

1. LangSmith가 필요한 이유
2. LLM 앱 디버깅이 어려운 이유
3. LangSmith 핵심 개념
4. 설치와 환경 변수 설정
5. LangChain 체인 추적하기
6. RAG 실행 흐름 추적하기
7. DB 챗봇 실행 흐름 추적하기
8. LangGraph 실행 흐름 추적하기
9. Dataset과 Example 개념
10. 평가와 품질 개선
11. VS Code와 Jupyter Notebook에서 사용할 때의 주의점
12. 오류 해결과 운영 팁

이 장은 `LLM/llm.ipynb`의 LangChain, RAG, Gradio 실습과 `LLM/llm2.ipynb`의 DB 챗봇 실습을 실제 개발 과정에서 추적하고 개선하는 단계에 해당합니다.


## 14.1 LangSmith가 필요한 이유

LLM 앱은 입력과 출력만 보면 내부에서 어떤 일이 일어났는지 알기 어렵습니다.

예를 들어 RAG 챗봇이 틀린 답변을 했을 때 원인은 여러 가지일 수 있습니다.

- 질문과 관련 없는 문서가 검색되었을 수 있습니다.
- 문서는 잘 검색되었지만 프롬프트가 좋지 않았을 수 있습니다.
- LLM이 검색 근거를 무시하고 답변했을 수 있습니다.
- 출력 파서가 응답을 잘못 처리했을 수 있습니다.
- 모델 설정값이 답변 안정성에 맞지 않았을 수 있습니다.

DB 챗봇도 마찬가지입니다.

- 스키마 정보가 잘못 전달되었을 수 있습니다.
- LLM이 잘못된 SQL을 만들었을 수 있습니다.
- SQL은 맞지만 DB 연결이나 권한 문제가 있을 수 있습니다.
- 조회 결과를 자연어로 설명하는 과정에서 해석이 틀렸을 수 있습니다.

LangSmith는 이런 단계별 실행 기록을 남겨서 문제를 찾고 개선할 수 있게 합니다.


## 14.2 LangSmith 핵심 개념

| 개념 | 의미 |
|---|---|
| Trace | 하나의 전체 실행 기록 |
| Run | Trace 안에 포함된 개별 실행 단계 |
| Project | 실행 기록을 묶어 관리하는 단위 |
| Dataset | 평가용 입력과 정답 예시 모음 |
| Example | Dataset 안의 한 개 테스트 사례 |
| Evaluator | 모델 답변을 평가하는 기준 또는 함수 |

Trace는 전체 실행 흐름입니다. 예를 들어 사용자가 질문 하나를 입력했을 때, 문서 검색, 프롬프트 생성, LLM 호출, 파싱, 최종 출력까지 하나의 Trace로 볼 수 있습니다.

Run은 Trace 안의 작은 실행 단위입니다. RAG에서는 retriever 호출, LLM 호출, output parser 호출이 각각 Run으로 기록될 수 있습니다.


In [ ]:
# 교재 위치: 14장 LangSmith - 설치 준비
# 이 셀은 LangSmith 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# langsmith는 LangSmith 추적, 데이터셋, 평가 기능을 사용하기 위한 라이브러리입니다.
!pip install langsmith

# langchain은 LangSmith와 함께 체인 실행을 추적할 때 사용하는 기본 프레임워크입니다.
!pip install langchain

# langchain-ollama는 Ollama 로컬 LLM을 LangChain 인터페이스로 호출할 때 사용합니다.
!pip install langchain-ollama

# python-dotenv는 .env 파일에 저장된 환경 변수를 노트북에서 불러올 때 사용합니다.
!pip install python-dotenv


## 14.3 환경 변수 설정

LangSmith 추적을 사용하려면 보통 다음 환경 변수를 설정합니다.

| 환경 변수 | 의미 |
|---|---|
| LANGCHAIN_TRACING_V2 | LangSmith 추적 사용 여부 |
| LANGCHAIN_ENDPOINT | LangSmith API 엔드포인트 |
| LANGCHAIN_API_KEY | LangSmith API 키 |
| LANGCHAIN_PROJECT | 기록을 저장할 프로젝트 이름 |

VS Code와 Jupyter Notebook에서는 환경 변수를 코드 셀에서 직접 지정하거나 `.env` 파일로 관리할 수 있습니다.

API 키는 노트북에 직접 적어 공유하지 않는 것이 좋습니다. 수업이나 실습 자료에서는 `.env` 파일을 사용하고, `.gitignore`에 `.env`를 추가하는 방식이 안전합니다.


In [ ]:
# 교재 위치: 14장 LangSmith - 환경 변수 설정
# 이 셀은 LangSmith 추적에 필요한 환경 변수를 설정하는 예시입니다.

# os 모듈은 운영체제 환경 변수에 접근할 때 사용합니다.
import os

# load_dotenv는 .env 파일에 저장된 값을 현재 Python 환경으로 불러옵니다.
from dotenv import load_dotenv

# 현재 작업 폴더 또는 상위 폴더의 .env 파일을 찾아 환경 변수로 로드합니다.
load_dotenv()

# LangSmith 추적 기능을 활성화합니다.
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# LangSmith API 엔드포인트를 지정합니다.
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# LangSmith 프로젝트 이름을 지정합니다.
os.environ["LANGCHAIN_PROJECT"] = "llm-basic-practice"

# LANGCHAIN_API_KEY는 .env 파일에서 읽어오는 것을 권장합니다.
# 예시 .env 내용: LANGCHAIN_API_KEY=여기에_발급받은_키
langsmith_api_key = os.getenv("LANGCHAIN_API_KEY")

# API 키가 설정되어 있는지 여부만 확인합니다.
print("LANGCHAIN_API_KEY 설정 여부:", langsmith_api_key is not None)


## 14.4 LangChain 체인 추적하기

LangSmith는 LangChain 체인 실행을 자동으로 추적할 수 있습니다. 환경 변수가 올바르게 설정되어 있으면 `prompt | llm | parser` 같은 체인을 실행할 때 LangSmith 프로젝트에 실행 기록이 남습니다.

아래 예제는 앞 장들에서 반복해서 사용한 기본 체인 구조입니다.

```text
ChatPromptTemplate -> ChatOllama -> StrOutputParser
```


In [ ]:
# 교재 위치: 14장 LangSmith - LangChain 체인 추적
# 이 셀은 간단한 LangChain 체인을 만들고 실행하는 예시입니다.

# ChatPromptTemplate은 채팅 모델에 전달할 메시지 템플릿을 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 채팅 모델의 출력 객체를 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama 로컬 모델을 LangChain 방식으로 호출합니다.
from langchain_ollama import ChatOllama

# 사용할 로컬 LLM 모델을 지정합니다.
llm = ChatOllama(model="llama3.1")

# 시스템 메시지와 사용자 메시지를 포함한 프롬프트 템플릿을 만듭니다.
prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 모델의 역할을 지정합니다.
    ("system", "너는 LLM 기초 교재를 설명하는 한국어 튜터야."),
    # human 메시지는 사용자의 질문을 question 변수로 받습니다.
    ("human", "다음 질문에 간단히 답해줘: {question}"),
])

# 프롬프트, LLM, 출력 파서를 파이프 연산자로 연결합니다.
chain = prompt | llm | StrOutputParser()

# 체인을 실행하면 LangSmith 설정이 켜져 있을 때 실행 Trace가 기록됩니다.
answer = chain.invoke({"question": "LangSmith는 왜 필요한가요?"})

# 생성된 답변을 출력합니다.
print(answer)


## 14.5 실행 이름과 태그 붙이기

LLM 앱을 여러 번 실행하면 기록이 많아집니다. 이때 실행 이름, 태그, 메타데이터를 붙이면 나중에 찾기 쉽습니다.

예를 들어 RAG 실습, DB 챗봇 실습, LangGraph 실습을 구분하려면 다음처럼 태그를 붙일 수 있습니다.

- `rag`
- `db-chatbot`
- `langgraph`
- `chapter-14`

LangChain의 `with_config()`를 사용하면 실행 이름과 태그를 지정할 수 있습니다.


In [ ]:
# 교재 위치: 14장 LangSmith - 실행 이름과 태그
# 이 셀은 LangChain 체인 실행에 이름과 태그를 붙이는 예시입니다.

# with_config는 체인 실행에 run_name, tags, metadata 같은 설정을 추가합니다.
tagged_chain = chain.with_config(
    # run_name은 LangSmith 화면에서 표시될 실행 이름입니다.
    {"run_name": "chapter14_basic_chain", "tags": ["chapter-14", "basic-chain"]}
)

# 태그가 붙은 체인을 실행합니다.
tagged_answer = tagged_chain.invoke({"question": "Trace와 Run의 차이를 설명해줘."})

# 실행 결과를 출력합니다.
print(tagged_answer)


## 14.6 RAG 실행 흐름 추적하기

RAG는 LangSmith로 추적할 때 특히 유용합니다. 답변이 틀렸을 때 검색이 문제인지, 프롬프트가 문제인지, 모델 응답이 문제인지 나눠서 볼 수 있기 때문입니다.

RAG Trace에서 확인할 내용은 다음과 같습니다.

- 사용자 질문이 그대로 들어갔는가
- Retriever가 어떤 문서를 가져왔는가
- 검색 문서가 질문과 실제로 관련 있는가
- Context가 프롬프트에 잘 들어갔는가
- LLM 답변이 Context를 근거로 작성되었는가
- 최종 출력 파서가 결과를 잘 처리했는가


In [ ]:
# 교재 위치: 14장 LangSmith - RAG 추적용 함수
# 이 셀은 RAG 흐름을 함수로 나누고 각 단계를 추적하기 쉽게 구성합니다.

# traceable은 일반 Python 함수도 LangSmith 추적 대상으로 만들 수 있는 데코레이터입니다.
from langsmith import traceable


# retrieve_mock_documents 함수는 문서 검색 단계를 흉내 내는 함수입니다.
@traceable(name="retrieve_mock_documents")
def retrieve_mock_documents(question: str) -> list[str]:
    # 실제 프로젝트에서는 FAISS나 Chroma retriever를 호출합니다.
    documents = [
        # 첫 번째 검색 결과 예시입니다.
        "RAG는 검색된 외부 문서를 근거로 LLM 답변을 생성하는 방식입니다.",
        # 두 번째 검색 결과 예시입니다.
        "Retriever는 질문과 의미적으로 가까운 문서 조각을 벡터스토어에서 찾습니다.",
    ]

    # 검색 결과 문서 목록을 반환합니다.
    return documents


# format_context 함수는 검색 문서를 하나의 문자열로 정리합니다.
@traceable(name="format_context")
def format_context(documents: list[str]) -> str:
    # 문서 목록을 줄바꿈으로 연결합니다.
    context = "\n".join(documents)

    # 정리된 context 문자열을 반환합니다.
    return context


# run_mock_rag 함수는 검색, 정리, 답변 생성을 하나로 연결합니다.
@traceable(name="run_mock_rag")
def run_mock_rag(question: str) -> str:
    # 질문을 사용해 관련 문서를 검색합니다.
    documents = retrieve_mock_documents(question)

    # 검색 문서를 프롬프트에 넣기 쉬운 context로 정리합니다.
    context = format_context(documents)

    # LLM에 전달할 입력 딕셔너리를 구성합니다.
    chain_input = {"question": question, "context": context}

    # RAG 답변용 프롬프트를 만듭니다.
    rag_prompt = ChatPromptTemplate.from_messages([
        # system 메시지는 근거 기반 답변 원칙을 지정합니다.
        ("system", "주어진 근거만 사용해서 한국어로 답변해."),
        # human 메시지는 질문과 검색 근거를 함께 전달합니다.
        ("human", "질문: {question}\n\n근거:\n{context}\n\n답변:"),
    ])

    # RAG 프롬프트, LLM, 출력 파서를 연결합니다.
    rag_chain = rag_prompt | llm | StrOutputParser()

    # RAG 체인을 실행하고 답변을 생성합니다.
    answer = rag_chain.invoke(chain_input)

    # 생성된 답변을 반환합니다.
    return answer


In [ ]:
# 교재 위치: 14장 LangSmith - RAG 추적 실행
# 이 셀은 RAG 흐름을 실행해 LangSmith Trace를 남기는 예시입니다.

# RAG 예시 질문을 문자열로 준비합니다.
rag_question = "RAG에서 Retriever는 어떤 역할을 하나요?"

# run_mock_rag 함수를 실행하면 내부 단계가 LangSmith에 함께 기록됩니다.
rag_answer = run_mock_rag(rag_question)

# RAG 답변을 출력합니다.
print(rag_answer)


## 14.7 DB 챗봇 실행 흐름 추적하기

DB 챗봇에서는 자연어 질문이 SQL로 바뀌는 과정이 중요합니다. LangSmith로 추적하면 다음을 확인할 수 있습니다.

- 사용자 질문이 어떤 SQL 생성 프롬프트에 들어갔는가
- DB 스키마가 정확히 전달되었는가
- LLM이 생성한 SQL은 무엇인가
- SQL 검증 단계에서 어떤 판단을 했는가
- 실행 결과는 몇 행이고 어떤 값인가
- 최종 자연어 답변은 조회 결과를 올바르게 설명하는가

운영 환경에서는 SQL 실행 전 검증 단계의 Trace를 반드시 남기는 것이 좋습니다.


In [ ]:
# 교재 위치: 14장 LangSmith - DB 챗봇 추적 함수
# 이 셀은 DB 챗봇의 주요 단계를 traceable 함수로 나눈 예시입니다.

# get_schema_mock 함수는 DB 스키마 조회 단계를 흉내 냅니다.
@traceable(name="get_schema_mock")
def get_schema_mock() -> str:
    # 실제 프로젝트에서는 SQLAlchemy inspector나 information_schema를 조회합니다.
    schema = "customers(id, name, age), orders(id, customer_id, amount, order_date)"

    # 스키마 문자열을 반환합니다.
    return schema


# generate_sql_mock 함수는 자연어 질문을 SQL로 변환하는 단계를 흉내 냅니다.
@traceable(name="generate_sql_mock")
def generate_sql_mock(question: str, schema: str) -> str:
    # 실제 프로젝트에서는 question과 schema를 LLM 프롬프트에 넣어 SQL을 생성합니다.
    sql = "SELECT customer_id, SUM(amount) AS total_amount FROM orders GROUP BY customer_id;"

    # 생성된 SQL 문자열을 반환합니다.
    return sql


# validate_sql_mock 함수는 SQL이 안전한 SELECT 문인지 검사합니다.
@traceable(name="validate_sql_mock")
def validate_sql_mock(sql: str) -> bool:
    # SQL을 소문자로 바꿔 키워드 검사를 쉽게 합니다.
    lowered_sql = sql.lower()

    # 실행을 막아야 하는 위험 키워드 목록입니다.
    blocked_keywords = ["insert", "update", "delete", "drop", "alter", "truncate"]

    # 위험 키워드가 포함되어 있는지 검사합니다.
    has_blocked_keyword = any(keyword in lowered_sql for keyword in blocked_keywords)

    # SELECT로 시작하고 위험 키워드가 없으면 안전하다고 판단합니다.
    is_safe = lowered_sql.strip().startswith("select") and not has_blocked_keyword

    # 안전성 판단 결과를 반환합니다.
    return is_safe


# execute_sql_mock 함수는 SQL 실행 단계를 흉내 냅니다.
@traceable(name="execute_sql_mock")
def execute_sql_mock(sql: str) -> str:
    # 실제 프로젝트에서는 pandas.read_sql 또는 connection.execute를 사용합니다.
    result = "customer_id=1, total_amount=150000\ncustomer_id=2, total_amount=90000"

    # 조회 결과 문자열을 반환합니다.
    return result


# run_db_chatbot_mock 함수는 DB 챗봇 전체 흐름을 실행합니다.
@traceable(name="run_db_chatbot_mock")
def run_db_chatbot_mock(question: str) -> str:
    # DB 스키마 정보를 가져옵니다.
    schema = get_schema_mock()

    # 질문과 스키마를 바탕으로 SQL을 생성합니다.
    sql = generate_sql_mock(question, schema)

    # 생성된 SQL이 안전한지 검사합니다.
    is_safe = validate_sql_mock(sql)

    # SQL이 안전하지 않으면 실행하지 않고 안내 메시지를 반환합니다.
    if not is_safe:
        # 안전하지 않은 SQL에 대한 안내 답변입니다.
        return "생성된 SQL이 안전하지 않아 실행하지 않았습니다."

    # 안전한 SQL이면 실행 결과를 가져옵니다.
    result = execute_sql_mock(sql)

    # 조회 결과를 자연어 답변으로 구성합니다.
    answer = f"질문: {question}\n생성 SQL: {sql}\n조회 결과:\n{result}"

    # 최종 답변을 반환합니다.
    return answer


In [ ]:
# 교재 위치: 14장 LangSmith - DB 챗봇 추적 실행
# 이 셀은 DB 챗봇 예시 흐름을 실행합니다.

# DB 챗봇에 전달할 자연어 질문을 준비합니다.
db_question = "고객별 총 주문 금액을 알려줘"

# DB 챗봇 흐름을 실행하면 각 단계가 LangSmith에 기록됩니다.
db_answer = run_db_chatbot_mock(db_question)

# 최종 답변을 출력합니다.
print(db_answer)


## 14.8 LangGraph 실행 흐름 추적하기

LangGraph는 여러 노드가 연결된 구조이므로 LangSmith와 함께 사용하면 각 노드의 입력과 출력을 추적하기 좋습니다.

13장에서 만든 LangGraph 흐름을 추적하면 다음을 확인할 수 있습니다.

- 질문 분류 노드가 어떤 route를 선택했는가
- RAG 경로로 갔는가, DB 경로로 갔는가, 일반 답변 경로로 갔는가
- 각 노드가 State의 어떤 값을 채웠는가
- 조건 분기 결과가 의도와 일치하는가
- 최종 State에 answer와 error가 어떻게 남았는가

복잡한 Agent를 만들수록 LangSmith의 Trace는 설계도처럼 중요해집니다.


In [ ]:
# 교재 위치: 14장 LangSmith - LangGraph 추적 개념 예시
# 이 셀은 LangGraph 실행을 traceable 함수로 감싸는 예시입니다.

# run_graph_with_trace 함수는 LangGraph 실행 전체를 하나의 추적 대상으로 만듭니다.
@traceable(name="run_graph_with_trace")
def run_graph_with_trace(question: str) -> dict:
    # 13장에서 만든 ChatState 형식에 맞춰 입력 State를 구성합니다.
    input_state = {
        # question에는 사용자의 질문을 넣습니다.
        "question": question,
        # route는 그래프의 질문 분류 노드가 채웁니다.
        "route": None,
        # context는 RAG나 DB 노드가 채웁니다.
        "context": None,
        # answer는 답변 생성 노드가 채웁니다.
        "answer": None,
        # error는 오류가 발생했을 때 채웁니다.
        "error": None,
    }

    # 실제 사용 시에는 13장에서 컴파일한 chat_graph.invoke(input_state)를 호출합니다.
    result_state = {
        # 예시에서는 route 값을 직접 넣어 결과 형태만 보여줍니다.
        "route": "rag",
        # 예시 context입니다.
        "context": "LangGraph는 State 기반 그래프 실행 도구입니다.",
        # 예시 answer입니다.
        "answer": "LangGraph는 복잡한 LLM 실행 흐름을 노드와 엣지로 관리합니다.",
        # 예시 error 값입니다.
        "error": None,
    }

    # 최종 State를 반환합니다.
    return result_state


# LangGraph 추적 예시 함수를 실행합니다.
graph_trace_result = run_graph_with_trace("문서 검색이 필요한 질문을 처리해줘")

# 그래프 실행 결과를 출력합니다.
print(graph_trace_result)


## 14.9 Dataset과 Example

LLM 앱을 한두 번 실행해 보는 것만으로는 품질을 판단하기 어렵습니다. 여러 질문을 모아 반복적으로 테스트해야 합니다. LangSmith의 Dataset은 이런 평가용 질문과 정답 예시를 관리하는 단위입니다.

Dataset에는 여러 Example이 들어갑니다. Example은 하나의 테스트 사례입니다.

예를 들어 RAG 챗봇 평가 Dataset에는 다음 예시를 넣을 수 있습니다.

| 입력 질문 | 기대 답변 |
|---|---|
| RAG란 무엇인가요? | 검색증강생성의 정의와 검색 후 생성 흐름 |
| 벡터스토어는 왜 필요한가요? | 임베딩 저장과 유사도 검색 설명 |
| FAISS와 Chroma 차이는? | 로컬 검색 엔진과 벡터 DB 관점 비교 |

DB 챗봇 평가 Dataset에는 다음 예시를 넣을 수 있습니다.

| 입력 질문 | 기대 SQL 또는 기대 답변 |
|---|---|
| 고객별 주문 건수를 알려줘 | GROUP BY customer_id 사용 |
| 월별 매출 합계를 알려줘 | 날짜 컬럼 기준 GROUP BY 사용 |
| 주문 금액 상위 5명을 알려줘 | ORDER BY amount DESC LIMIT 5 사용 |


In [ ]:
# 교재 위치: 14장 LangSmith - 평가 데이터 예시
# 이 셀은 Dataset에 넣을 수 있는 평가 예시 데이터를 Python 리스트로 표현합니다.

# rag_examples는 RAG 챗봇 평가에 사용할 질문과 기대 답변 목록입니다.
rag_examples = [
    # 첫 번째 RAG 평가 예시입니다.
    {"question": "RAG란 무엇인가요?", "expected": "검색증강생성의 정의와 검색 후 생성 흐름을 설명해야 합니다."},
    # 두 번째 RAG 평가 예시입니다.
    {"question": "벡터스토어는 왜 필요한가요?", "expected": "임베딩 저장과 유사도 검색의 필요성을 설명해야 합니다."},
    # 세 번째 RAG 평가 예시입니다.
    {"question": "FAISS와 Chroma의 차이는 무엇인가요?", "expected": "FAISS와 Chroma의 저장, 검색, 활용 차이를 설명해야 합니다."},
]

# db_examples는 DB 챗봇 평가에 사용할 질문과 기대 SQL 특징 목록입니다.
db_examples = [
    # 첫 번째 DB 평가 예시입니다.
    {"question": "고객별 주문 건수를 알려줘", "expected_sql_hint": "GROUP BY customer_id"},
    # 두 번째 DB 평가 예시입니다.
    {"question": "월별 매출 합계를 알려줘", "expected_sql_hint": "GROUP BY month 또는 날짜 함수"},
    # 세 번째 DB 평가 예시입니다.
    {"question": "주문 금액 상위 5명을 알려줘", "expected_sql_hint": "ORDER BY amount DESC LIMIT 5"},
]

# RAG 평가 예시를 출력합니다.
print(rag_examples)

# DB 평가 예시를 출력합니다.
print(db_examples)


## 14.10 간단한 평가 함수 만들기

LangSmith에서는 평가기를 사용해 모델 답변을 비교할 수 있습니다. 실습 단계에서는 간단한 Python 함수로도 평가 기준을 만들 수 있습니다.

예를 들어 다음 기준을 사용할 수 있습니다.

- 답변이 비어 있지 않은가
- 기대 키워드가 포함되어 있는가
- SQL에 위험 키워드가 없는가
- RAG 답변이 검색 근거를 언급하는가

실제 운영에서는 정확성, 근거 충실성, 유해성, 형식 준수, 응답 시간 같은 지표를 함께 봐야 합니다.


In [ ]:
# 교재 위치: 14장 LangSmith - 간단한 평가 함수
# 이 셀은 답변과 SQL을 간단히 평가하는 함수 예시입니다.

# evaluate_non_empty 함수는 답변이 비어 있지 않은지 평가합니다.
def evaluate_non_empty(answer: str) -> bool:
    # answer가 문자열이고 공백을 제거했을 때 길이가 0보다 큰지 확인합니다.
    return isinstance(answer, str) and len(answer.strip()) > 0


# evaluate_keyword 함수는 답변에 기대 키워드가 들어 있는지 평가합니다.
def evaluate_keyword(answer: str, keyword: str) -> bool:
    # 답변과 키워드를 소문자로 바꿔 포함 여부를 검사합니다.
    return keyword.lower() in answer.lower()


# evaluate_safe_sql 함수는 SQL이 읽기 전용 SELECT 문인지 평가합니다.
def evaluate_safe_sql(sql: str) -> bool:
    # SQL을 소문자로 바꿔 검사합니다.
    lowered_sql = sql.lower()

    # 위험 키워드 목록을 정의합니다.
    blocked_keywords = ["insert", "update", "delete", "drop", "alter", "truncate"]

    # 위험 키워드 포함 여부를 확인합니다.
    has_blocked_keyword = any(keyword in lowered_sql for keyword in blocked_keywords)

    # SELECT로 시작하고 위험 키워드가 없으면 True를 반환합니다.
    return lowered_sql.strip().startswith("select") and not has_blocked_keyword


# 예시 답변을 준비합니다.
sample_answer = "RAG는 검색증강생성으로, 외부 문서를 검색한 뒤 답변을 생성합니다."

# 예시 SQL을 준비합니다.
sample_sql = "SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id;"

# 답변이 비어 있지 않은지 평가합니다.
print("답변 비어 있지 않음:", evaluate_non_empty(sample_answer))

# 답변에 RAG 키워드가 포함되어 있는지 평가합니다.
print("RAG 키워드 포함:", evaluate_keyword(sample_answer, "RAG"))

# SQL이 안전한지 평가합니다.
print("SQL 안전성:", evaluate_safe_sql(sample_sql))


## 14.11 LangSmith를 이용한 개선 흐름

LangSmith는 단순히 로그를 보는 도구가 아니라, LLM 앱을 반복 개선하는 데 사용합니다.

기본 개선 흐름은 다음과 같습니다.

```text
1. 실습 앱 실행
2. LangSmith Trace 확인
3. 실패 사례 수집
4. Dataset에 실패 사례 추가
5. 프롬프트, retriever, 모델 설정 수정
6. 같은 Dataset으로 다시 평가
7. 이전 결과와 비교
```

이 흐름은 RAG, DB 챗봇, LangGraph Agent 모두에 적용할 수 있습니다.


## 14.12 VS Code와 Jupyter Notebook 사용 팁

VS Code에서 Jupyter Notebook으로 LangSmith를 사용할 때는 다음을 확인합니다.

- 노트북 커널이 패키지가 설치된 Python 환경을 사용하고 있는지 확인합니다.
- `.env` 파일이 현재 작업 폴더에서 잘 읽히는지 확인합니다.
- API 키를 코드 셀에 직접 적지 않습니다.
- 노트북을 공유할 때 출력 결과에 민감 정보가 남아 있지 않은지 확인합니다.
- 프로젝트 이름을 장별로 다르게 지정하면 기록을 구분하기 쉽습니다.
- Ollama 모델을 사용하는 경우 Ollama 서버가 실행 중이어야 합니다.

교재 실습에서는 `LANGCHAIN_PROJECT` 값을 다음처럼 구분할 수 있습니다.

```text
llm-basic-rag
llm-basic-db-chatbot
llm-basic-langgraph
llm-basic-evaluation
```


## 14.13 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| Trace가 보이지 않음 | 환경 변수 미설정 | LANGCHAIN_TRACING_V2, API KEY, PROJECT 확인 |
| 인증 오류 | API 키 오류 | LangSmith API 키 재발급 또는 .env 확인 |
| 프로젝트가 섞임 | 같은 프로젝트 이름 사용 | 장별 또는 앱별 프로젝트 이름 사용 |
| LLM 호출 실패 | Ollama 서버 미실행 | Ollama 실행 후 모델 이름 확인 |
| RAG 품질 낮음 | 검색 문서 부적합 | 청크 크기, 임베딩 모델, retriever 설정 점검 |
| DB 챗봇 오류 | SQL 생성 또는 실행 실패 | 스키마 전달, SQL 검증, DB 연결 확인 |
| 평가 결과 불안정 | 평가 기준 모호 | 기대 답변, 키워드, 평가 함수를 명확히 정의 |


## 14.14 정리

이 장에서는 LangSmith를 사용해 LLM 애플리케이션을 추적하고 개선하는 방법을 정리했습니다.

핵심 정리는 다음과 같습니다.

- LangSmith는 LLM 앱의 실행 과정을 Trace와 Run 단위로 기록합니다.
- RAG에서는 검색 문서, 프롬프트, LLM 답변을 단계별로 확인할 수 있습니다.
- DB 챗봇에서는 SQL 생성, 검증, 실행, 결과 해석 과정을 확인할 수 있습니다.
- LangGraph에서는 노드별 입력과 출력, 조건 분기 흐름을 추적할 수 있습니다.
- Dataset과 Example을 사용하면 반복 평가가 가능합니다.
- 평가 결과를 바탕으로 프롬프트, retriever, 모델 설정, 그래프 구조를 개선할 수 있습니다.

다음 장에서는 STT와 TTS 보조 실습을 다룹니다. 음성을 텍스트로 바꾸고, 텍스트를 다시 음성으로 변환하는 방식은 LLM 챗봇을 음성 인터페이스로 확장할 때 사용됩니다.
